# Onboarding: llm-posttrain-lab on the GPU node

Run this top to bottom on a fresh pod (VS Code server / Kubeflow notebook) before your first training run.
Each section checks one thing, so you can re-run any cell on its own later.

**What this repo does:** compares SFT, DPO, PPO and GRPO on a reward-vs-KL frontier, on two tracks:
*helpful* (UltraFeedback, learned reward model) and *math* (GSM8K, verifiable reward). See `README.md`.

**Status levels:** `OK` · `INFO` (for the record) · `WARN` (works, but look at it) · `FAIL` (a pipeline stage will break).
The same checks run headless with `python -m posttrain_lab.preflight [check ...]` or `make preflight`.

## 1. Find the repo and import the checks

Works from any directory inside the repo. If you haven't installed the package yet, it's put on `sys.path` for this kernel.

In [ ]:
import sys, pathlib

ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "configs" / "base.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "configs" / "base.yaml").exists(), "run this notebook from inside the llm-posttrain-lab repo"
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from posttrain_lab import preflight
print("repo:", ROOT)
preflight.list_checks()

## 2. Install dependencies

Only needed once per image/venv. The cluster has no internet, so this goes through your internal package mirror
(set `PIP_INDEX_URL` / `pip.conf` if it isn't already). The pins in `requirements.txt` are load-bearing:
TRL 1.7 ↔ vLLM 0.23 ↔ transformers 5.13 were verified together.

Uncomment and run, then **restart the kernel**.

In [ ]:
# %pip install -r ../requirements.txt
# %pip install -e .. --no-deps

In [ ]:
preflight.run("packages")

## 3. Configure the MLflow endpoint (`.env`)

Training and eval log to MLflow. The tracking URI is read, in order of precedence, from:
`mlflow.tracking_uri=...` on the command line → `MLFLOW_TRACKING_URI` in the pod env → `MLFLOW_TRACKING_URI` in the repo's `.env`.
`MLFLOW_PORT` is appended when the URI from the same source has no port. `.env` is git-ignored.

The next cell creates `.env` from `.env.example` **only if it doesn't exist**. Then fill in `MLFLOW_TRACKING_URI`,
and set `MLFLOW_PORT` (e.g. `5000`) only if the server is on a non-standard port that the URI doesn't include.
Check the resolved URI the cell prints.

In [ ]:
import shutil
env_file, example = ROOT / ".env", ROOT / ".env.example"
if not env_file.exists():
    shutil.copy(example, env_file)
    print(f"created {env_file} from .env.example — edit MLFLOW_TRACKING_URI, then re-run this cell")
print(env_file.read_text())

from posttrain_lab.config import load_config
from posttrain_lab.utils.logging import _tracking_uri
print("resolved tracking URI ->", _tracking_uri(load_config("helpful/stage1_sft_base.yaml", [])))

## 4. Node resources

CPU/RAM, `/dev/shm` and free disk. Kubeflow pods default to a 64 MB `/dev/shm`, which breaks NCCL and dataloader
workers. If it's small, ask for an `emptyDir` with `medium: Memory` mounted at `/dev/shm`.
Results take ~250 GB (helpful) + ~60 GB (math), plus ~20 GB of models in the HF cache.

In [ ]:
preflight.run("system")

## 5. GPUs

Device list, bf16 support, a bf16 matmul benchmark per GPU, P2P access, and an NCCL all-reduce across all visible GPUs
(the same collective DeepSpeed ZeRO-2 uses). Takes ~30 s. Low matmul TFLOPS means a throttled or shared GPU;
low all-reduce bandwidth usually means no NVLink, which makes training slower but still works.

In [ ]:
preflight.run("gpu")

## 6. Training and inference libraries

* **flash_attn**: `configs/base.yaml` asks for `flash_attention_2`. If it isn't installed, either install it or set
  `model.attn_implementation: sdpa`.
* **deepspeed**: used for every multi-GPU stage (`configs/accelerate/zero2_*.yaml`).
* **vllm**: GRPO rollouts (colocated) and eval generation.

In [ ]:
preflight.run("flash_attn", "deepspeed", "vllm")

Optional, slower (~1 min): load the smoke model in vLLM and generate a few tokens. It needs the smoke model
available (section 7).

In [ ]:
# preflight.run("vllm_generate")

## 7. Network and Hugging Face assets

The cluster has no internet, so every model and dataset the pipeline loads must come from either an **internal mirror**
or a **pre-populated cache**. The `hf` check lists each asset (base, smoke and judge models; UltraFeedback, GSM8K,
wikitext and lm-eval's hellaswag / ARC data) as *in local cache*, *downloadable* or `FAIL` (that stage will crash).
Details: README, "Offline cluster".

**Option A: internal mirror / proxy.** Set `HF_ENDPOINT=https://<your-hf-mirror>` (plus `HF_TOKEN` if it needs one).
Optionally put `HF_HOME` on a PVC so downloads survive pod restarts.

**Option B: pre-populated cache on a PVC.** On a machine *with* internet, run
`python scripts/prefetch_assets.py /data/hf_home` (or `--smoke` for just the smoke assets, `--skip judge` to leave out
the 8B judge); it downloads everything and then verifies the directory offline. Copy it to the PVC with symlinks intact
(`rsync -a` or `tar`), then set `HF_HOME=/mnt/hf HF_HUB_OFFLINE=1 HF_DATASETS_OFFLINE=1`. Both `hub/` and `datasets/`
must be in the copy; leave `HF_HUB_CACHE` / `HF_DATASETS_CACHE` unset.

Put the variables in the pod spec or in `.env` (commented templates in `.env.example`). HF libraries read them at
import time, so **restart the kernel** after changing them. The next cell shows what this kernel sees.

`internet` is informational only: on this cluster, unreachable is the expected result.

In [ ]:
import os
for var in ("HF_HOME", "HF_ENDPOINT", "HF_HUB_OFFLINE", "HF_DATASETS_OFFLINE", "HF_HUB_CACHE", "HF_DATASETS_CACHE"):
    print(f"{var:20s} {os.environ.get(var, '<unset>')}")
preflight.run("internet", "hf")

Optional (~1 min): actually load every asset with the current env (tokenizer + config per model, every dataset split,
lm-eval's own task build). Catches a cache that is present but unusable, e.g. a copy missing `datasets/`.

In [ ]:
# preflight.run("hf_load")

## 8. MLflow end to end

Hits `/health` and `/version`, then creates a test run in experiment `llm-posttrain-lab-preflight`, writes 3 metric
points and reads them back. Open the printed link to check the UI. Real runs land in `llm-posttrain-lab-helpful` /
`llm-posttrain-lab-math`. Each run carries tags `arm`, `hyperparam`, `seed` and `run_dir`; training metrics are logged
per step and eval metrics as `eval/*` at each checkpoint's step.

In [ ]:
preflight.run("mlflow")

## 9. Everything at once

Re-run after fixing things. Same as `make preflight` in a terminal.

In [ ]:
ok = preflight.run()
print("READY" if ok else "fix the FAILs above before training")

## 10. Next steps

1. **Smoke test** (~30 min, 1 GPU, tiny model, tracking off): `make smoke`. It runs every stage interface end to end.
2. **Real runs**, in order (see the README table): `make helpful-all` and/or `make math-all`, or stage by stage:
   `make sft-base` → `make rm` → `make arm-sft arm-dpo` → `make arm-ppo arm-grpo` → `make eval` → `make plots`.
3. **Watch progress** in MLflow (experiment `llm-posttrain-lab-<task>`), or follow the logs:
   the DPO sweep writes `logs_dpo_*.log` and eval writes `logs_eval_shard*.log` in the repo root.